In [1]:
#manupulação de dados em tabelas
import pandas as pd 
#plots de gráficos
import matplotlib.pyplot as plt
#manupalação de vetores
import numpy as np 

#biblioteca spacy
import spacy
#OBS: Caso não tenha o pacote em português, execute no terminal o comando: python3 -m spacy download pt

# biblioteca string - Nativa do python
import string 

#Os stop words são oriundo da biblioteca spacy
from spacy.lang.pt.stop_words import STOP_WORDS

In [2]:
pln=spacy.load("pt_core_news_sm")
stop_words=STOP_WORDS
pontuacoes=string.punctuation
pontuacoes=pontuacoes+"..."+' '

# remove da lista de stop words alguns elementos importantes
stop_words.remove('bom')
stop_words.remove('muito')
stop_words.remove('não')
stop_words.remove('nem')

In [3]:
def processamento(texto):
    # texto em minuscula
    texto=texto.lower()
    documento=pln(texto)
    
    #removendo stop words
    lista_tokens_1=[]
    for p in documento:
        if (p.text in stop_words)==False:
            lista_tokens_1.append(p)
    #removendo pontuações      
    lista_tokens_2=[]
    for p in lista_tokens_1:
        if (p.text in pontuacoes)==False:
            lista_tokens_2.append(p)
    #lematização de tokens        
    lista_tokens_3=[]
    for p in lista_tokens_2:
        lista_tokens_3.append(p.lemma_)

    return lista_tokens_3

In [7]:
df=pd.read_csv('olist_order_reviews_dataset.csv')
# Amostrado da tabela
df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [8]:
df['review_creation_date']=pd.to_datetime(df['review_creation_date'])
# encontrando o ano de publicação
ano=[]
for i in range(len(df)):
    ano.append(df['review_creation_date'].iloc[i].year)
df['ANO']=ano

# Seleção de comentários de 2018
df=df[df['ANO']==2018].reset_index(drop=True)

#remover linhas duplicadas
df.drop_duplicates(subset='review_id',inplace=True)
# selecionar apenas algumas colunas releantes
df=df[['review_comment_title','review_comment_message','review_score']].reset_index(drop=True)
# preencher campos vazios
df.fillna('',inplace=True)
# reestruturação dos comentários
df['review']=df['review_comment_title']+ ' '+df['review_comment_message']
# remoção de comentário vazios
df['review']=df['review'].replace(' ',np.nan)
df=df.dropna(subset="review").reset_index(drop=True)

# utilizar uma amostra do dado 
df=df.sample(5000).reset_index(drop=True)

In [10]:
df.review_score.value_counts()

review_score
5    2519
1    1042
4     744
3     446
2     249
Name: count, dtype: int64

# Vetorizar textos

In [11]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import seaborn as sns 

In [12]:
corpus=list(df['review'].values)

In [13]:
# essa etapa demora um pouco
vectorizer = CountVectorizer(tokenizer=processamento,max_features=1000,stop_words=None,token_pattern=None)
vectorizer.fit(corpus)

CountVectorizer(max_features=1000, token_pattern=None,
                tokenizer=<function processamento at 0x7fea0c4ff280>)

In [14]:
vocabulario=vectorizer.get_feature_names()

/home/jlbdearaujo/.local/lib/python3.8/site-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


In [15]:
bow=vectorizer.transform(corpus)

In [16]:
d_bow=pd.DataFrame(data=bow.toarray(),columns=vocabulario)

# Método de árvore 

In [24]:
from sklearn.ensemble import RandomForestClassifier

In [25]:

X_bow=d_bow.values
y_score_bow=y=df['review_score'].values

In [26]:

rdf = RandomForestClassifier(random_state=42)
rdf.fit(X_bow, y_score_bow)


RandomForestClassifier(random_state=42)

In [28]:
rdf.feature_importances_.shape

(1000,)

# verificar a importancia 

In [32]:
# top 20 dos termos mais relevantes para classificação
pd.DataFrame({"FEATURES":d_bow.columns,"IMPORTANCIA":rdf.feature_importances_}).sort_values("IMPORTANCIA",ascending=False).head(20)

,FEATURES,IMPORTANCIA
625,não,0.067852
627,o,0.035597
752,produto,0.024529
797,receber,0.023943
333,e,0.018269
599,muito,0.017705
364,entregar,0.016871
805,recomendar,0.016144
724,prazo,0.015397
152,bom,0.014972
